# 实验二：房源采集与 MySQL 存储

采集安居客教学站（http://www.bspider.top/anjuke/ ）的房源信息，保存为 CSV，再写入 MySQL。

使用 requests、BeautifulSoup（导入名 `bs4`）和 PyMySQL，网页解析使用 CSS 选择器。

需要采集 8 个字段：`title, house, address, struct, area, floor, maketime, price`

In [ ]:
# 如缺少依赖，运行本单元格安装
%pip install requests beautifulsoup4 pymysql

## 第一阶段：MySQL 建表（在 MySQL 终端执行，不在 Notebook 中）

```sql
USE smartpark_lab;
CREATE TABLE IF NOT EXISTS houseinfo (
    id INT AUTO_INCREMENT PRIMARY KEY COMMENT '自增长主键',
    title VARCHAR(200) COMMENT '标题',
    house VARCHAR(50) COMMENT '楼盘',
    address VARCHAR(200) COMMENT '地址',
    struct VARCHAR(50) COMMENT '户型',
    area VARCHAR(20) COMMENT '建筑面积',
    floor VARCHAR(20) COMMENT '楼层',
    maketime VARCHAR(20) COMMENT '建造年代',
    price VARCHAR(20) COMMENT '每平方米单价'
) CHARACTER SET utf8mb4;
DESCRIBE houseinfo;
SELECT COUNT(*) AS before_count FROM houseinfo;
```

记下入库前条数 `before_count`，后面计算本次新增条数。

## 第二阶段：房源采集

### 1. 获取第一页

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "http://www.bspider.top/anjuke/"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

print(r.status_code)
print(soup.title.get_text(strip=True) if soup.title else "无标题")

# TODO: 检查标题/内容，确认不是访问验证页；若是验证页或无法访问，记录后停止并联系教师

### 2. 提取 8 个字段

房源容器为 `div.property`（先在当前页面核对）。先解析第一条逐字段核对，再处理整页。

In [ ]:
# TODO: 定位房源容器
rows = soup.select("div.property")
print("本页房源条数:", len(rows))

# TODO: 取 rows[0]，用 select_one 定位 h3 等字段，输出 8 个字段逐一对照网页
# 注意：
# - 地址、户型文字可能分散在多个子标签中，取完整文字
# - 面积/楼层先打印属性组确认顺序，不能把朝向当成楼层
# - 建造年代缺失时填空字符串（先判断节点是否存在）
# - 单价是每平方米单价，不是"万元"总价

### 3. 采集下一页

下一页按钮可用 `a.next.next-active` 定位，用 `.get("href")` 读链接，再用 `urljoin` 补全。

顺序：请求当前页 → 解析加入列表 → 找下一页并补全 → 检查地址不越界、不重复 → 等待至少 1 秒。

先试采 2 页确认无重复，再继续。设置安全页数上限防止无限循环。

In [ ]:
# TODO: 编写分页循环
# from urllib.parse import urljoin
# import time
#
# 记录：下一页 href 原值、urljoin 补全后的地址、每页页码/URL/本页条数/累计条数
# 停止条件：只有"下一页"按钮确认禁用才算到达末页；找不到按钮记为结构异常
# 结果列表放 all_houses（函数内部或循环内定义，不要放全局）

### 4. 保存和核对 CSV

保存到 `data/houseinfo.csv`：用 `csv.writer`，编码 `utf-8-sig`，`newline=""`；表头为 8 个字段名且顺序一致；按 8 个字段完全相同去重后保存，记录去重前后条数。再用 `csv.DictReader` 回读核对条数和样本。

In [ ]:
# TODO: CSV 保存（含去重）与 DictReader 回读核对
# import csv
# from pathlib import Path

## 第三阶段：MySQL 数据存储

读取 `data/houseinfo.csv` 写入 MySQL，不重新爬取。

连接参数：host=`mysql`，port=`3306`，database=`smartpark_lab`，user=`student`，password=`123456`，charset=`utf8mb4`，autocommit=`False`。

要求：`csv.DictReader` 读 CSV 并检查表头 → 连接并查询确认 → `executemany()` 用 `%s` 占位写入（不要字符串拼接）→ 成功 `commit()`，出错 `rollback()` → 最后先关游标再关连接。重跑时按 8 个字段比较已有记录，只插入不存在的记录，不要清空表。

In [ ]:
# TODO: 自己编写入库程序
# import pymysql
# 1. 读 CSV，按 8 字段固定顺序整理记录
# 2. pymysql.connect(...)，先查询一条确认连接
# 3. INSERT INTO houseinfo (title, house, address, struct, area, floor, maketime, price) VALUES (%s,...)
# 4. executemany + commit；出错 rollback；finally 中先关游标再关连接
# 5. 重跑防重复：比较库内已有记录

## 验收入库结果（MySQL 终端执行）

```sql
USE smartpark_lab;
DESCRIBE houseinfo;
SELECT COUNT(*) AS after_count FROM houseinfo;
SELECT title, house, address, struct, area, floor, maketime, price
FROM houseinfo ORDER BY id LIMIT 5;
```

记录入库前后条数与本次新增条数；重跑一次入库程序再查询，确认不重复增加；抽取 3 条与 CSV 逐字段核对。截图存入 `images/`，报告见 `REPORT.md`。